In [1]:
!pip install langchain langchain-community langchain-huggingface langchain-google-genai chromadb python-dotenv sentence-transformers


In [2]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
print("API cargada:", os.environ["GOOGLE_API_KEY"][:8], "*****")


API cargada: AIzaSyAn *****


In [3]:
from google.colab import drive
drive.mount('/content/drive')

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

# 1. Cargar documento desde Google Drive
loader = TextLoader("/content/drive/MyDrive/Colab Notebooks/w4-ai/docs/intro-to-llms-karpathy.txt")
documents = loader.load()

# 2. Dividir texto
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(documents)

# 3. Crear embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

# 4. Crear vector store persistente
persist_directory = "/content/drive/MyDrive/Colab Notebooks/w4-ai/db/karpathy_chroma"

vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory=persist_directory
)

vector_store.persist()

print("🔥 Base vectorial creada exitosamente.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


🔥 Base vectorial creada exitosamente.


/tmp/ipython-input-1668952336.py:29: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_store.persist()


In [4]:
# ============================================
# 🔵 FASE B – Pipeline RAG y preguntas de prueba
# ============================================

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from dotenv import load_dotenv
import os

# 1. Asegurar que la API Key está cargada
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# 2. Cargar el LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",   # 🔥 modelo estable para LangChain
    temperature=0.2
)

# 3. Cargar embeddings (mismo modelo que la fase A)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

# 4. Cargar la base vectorial desde el disco
persist_directory = "/content/drive/MyDrive/Colab Notebooks/w4-ai/db/karpathy_chroma"

vector_store = Chroma(
    persist_directory=persist_directory,
    embedding_function=embeddings
)

# 5. Crear el pipeline RAG
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vector_store.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True
)

# 6. Probar con una pregunta
question = "¿Qué es un transformer según Karpathy?"

response = qa_chain.invoke({"query": question})

print("🔵 RESPUESTA:\n", response["result"])
print("\n📚 CONTEXTOS USADOS:")
for i, doc in enumerate(response["source_documents"], start=1):
    print(f"\n--- Documento {i} ---\n")
    print(doc.page_content[:400], "...")

/tmp/ipython-input-3444394256.py:27: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vector_store = Chroma(


🔵 RESPUESTA:
 Según Karpathy, un transformer es una arquitectura de red neuronal de la cual entendemos a detalle su arquitectura y las operaciones matemáticas que ocurren en sus diferentes etapas.

📚 CONTEXTOS USADOS:

--- Documento 1 ---

with is as we call hallucination or like an incorrect answer or like a correct answer necessarily, so some of the stuff could be memorized and some of it is not memorized and you don't exactly know which is which um but for the most part this is just kind of like halucinating or like dreaming internet text from its data distribution. okay let's now switch gears to how does this network work? work, ...

--- Documento 2 ---

with is as we call hallucination or like an incorrect answer or like a correct answer necessarily, so some of the stuff could be memorized and some of it is not memorized and you don't exactly know which is which um but for the most part this is just kind of like halucinating or like dreaming internet text from its data distributio

In [6]:
# ahora lo aplicamos para las 50 preguntas
import json

# Cargar las 50 preguntas
questions_path = "/content/drive/MyDrive/Colab Notebooks/w4-ai/docs/questions.json"

with open(questions_path, "r", encoding="utf-8") as f:
    test_questions = json.load(f)

print("Total preguntas cargadas:", len(test_questions))
print(test_questions[:3])  # imprimir 3 para revisar


Total preguntas cargadas: 50
[{'question': 'What are some security challenges associated with large language models?', 'answer': '', 'contexts': []}, {'question': 'What is the purpose of the base model in the process of developing an assistant model?', 'answer': '', 'contexts': []}, {'question': 'What is an adversarial example in the context of large language models?', 'answer': '', 'contexts': []}]


In [7]:
from tqdm import tqdm

results = []

print("⚡ Generando respuestas con RAG...\n")

for q in tqdm(test_questions):
    question_text = q["question"]  # 👈 aquí usamos solo el texto de la pregunta

    response = qa_chain.invoke({"query": question_text})

    results.append({
        "question": question_text,
        "answer": response["result"],
        "contexts": [
            doc.page_content[:500]  # recortamos cada contexto para que el JSON no sea gigante
            for doc in response["source_documents"]
        ]
    })

print("🔥 ¡Terminó!")


⚡ Generando respuestas con RAG...



 32%|███▏      | 16/50 [00:27<01:00,  1.77s/it]WARNING:langchain_google_genai.chat_models:Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-2.0-flash
Please retry in 28.788073352s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.0-fl

🔥 ¡Terminó!


In [10]:
import json

path = "/content/drive/MyDrive/Colab Notebooks/w4-ai/my_rag_output.json"

with open(path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print("Archivo guardado correctamente en:", path)

Archivo guardado correctamente en: /content/drive/MyDrive/Colab Notebooks/w4-ai/my_rag_output.json


In [11]:
import os
print("Existe:", os.path.exists(path))

Existe: True
